<a href="https://colab.research.google.com/github/yeminaung-web/AI_STUDY/blob/main/UCI_HAR/Inception.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import urllib.request
import zipfile
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


In [2]:

# ============================================================
# 1. CONFIGURATION & REPRODUCIBILITY
# ============================================================
SEED = 42
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
MODEL_PATH = "best_inception_bilstm_uci_har.pt"
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/human+activity+recognition+using+smartphones/UCI HAR Dataset"

ACTIVITY_LABELS = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
]

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [3]:

# ============================================================
# 2. DOWNLOAD & LOAD UCI-HAR INERTIAL SIGNALS
# ============================================================
if not os.path.exists(DATASET_PATH):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
    urllib.request.urlretrieve(url, "uci_har.zip")
    with zipfile.ZipFile("uci_har.zip", "r") as zf:
        zf.extractall(".")

SIGNALS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]

def load_split(subset):
    signals_data = []
    for signal in SIGNALS:
        file_path = os.path.join(DATASET_PATH, subset, "Inertial Signals", f"{signal}_{subset}.txt")
        signals_data.append(np.loadtxt(file_path))
    # Transpose from (9, samples, 128) -> (samples, 128, 9)
    X = np.transpose(np.array(signals_data), (1, 2, 0)).astype(np.float32)
    y_path = os.path.join(DATASET_PATH, subset, f"y_{subset}.txt")
    y = (np.loadtxt(y_path, dtype=np.int64) - 1)  # Convert labels 1..6 -> 0..5
    return X, y

X_train_full, y_train_full = load_split("train")
X_test, y_test = load_split("test")

# Train / Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)

# Leakage-free StandardScaler (fit ONLY on X_train)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, 9)).reshape(X_train.shape)
X_val = scaler.transform(X_val.reshape(-1, 9)).reshape(X_val.shape)
X_test = scaler.transform(X_test.reshape(-1, 9)).reshape(X_test.shape)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val), torch.tensor(y_val)),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
    batch_size=BATCH_SIZE,
    shuffle=False,
)


In [4]:

# ============================================================
# 3. GOOGLENET (INCEPTION) + BiLSTM ARCHITECTURE
# ============================================================
class ConvBNReLU1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class InceptionBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        bf = out_channels // 4

        # Branch 1: 1x1 Conv
        self.branch1 = ConvBNReLU1D(in_channels, bf, kernel_size=1)

        # Branch 2: 1x1 Bottleneck -> 3x1 Conv
        self.branch2 = nn.Sequential(
            ConvBNReLU1D(in_channels, bf, kernel_size=1),
            ConvBNReLU1D(bf, bf, kernel_size=3, padding=1),
        )

        # Branch 3: 1x1 Bottleneck -> 7x1 Conv
        self.branch3 = nn.Sequential(
            ConvBNReLU1D(in_channels, bf, kernel_size=1),
            ConvBNReLU1D(bf, bf, kernel_size=7, padding=3),
        )

        # Branch 4: 3x1 MaxPool (stride=1) -> 1x1 Conv
        self.branch4 = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            ConvBNReLU1D(in_channels, bf, kernel_size=1),
        )

    def forward(self, x):
        return torch.cat(
            [self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)],
            dim=1,
        )


class InceptionBiLSTM(nn.Module):
    def __init__(self, in_channels=9, num_classes=6):
        super().__init__()
        self.inception1 = InceptionBlock1D(in_channels, 64)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout(0.30)

        self.inception2 = InceptionBlock1D(64, 128)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout(0.30)

        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            batch_first=True,
            bidirectional=True,
        )
        self.drop3 = nn.Dropout(0.30)

        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        # (batch, 128, 9) -> (batch, 9, 128) for Conv1d
        x = x.permute(0, 2, 1)

        x = self.drop1(self.pool1(self.inception1(x)))   # (batch, 64, 64)
        x = self.drop2(self.pool2(self.inception2(x)))   # (batch, 128, 32)

        # (batch, 128, 32) -> (batch, 32, 128) for LSTM
        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        # Concatenate last forward state h_n[-2] and last backward state h_n[-1]
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)         # (batch, 256)
        x = self.drop3(x)

        logits = self.classifier(x)
        return logits


model = InceptionBiLSTM(in_channels=9, num_classes=6).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")


Total params: 303,174 | Trainable: 303,174


In [5]:

# ============================================================
# 4. TRAINING LOOP (WITH LR SCHEDULER & EARLY STOPPING)
# ============================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

best_val_loss = float("inf")
best_epoch = 0
early_stop_patience = 15
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        train_correct += (logits.argmax(dim=1) == yb).sum().item()
        train_total += xb.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)
            val_correct += (logits.argmax(dim=1) == yb).sum().item()
            val_total += xb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f} | LR: {current_lr:.2e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"Early stopping at Epoch {epoch}. Best Epoch: {best_epoch} (Val Loss: {best_val_loss:.5f})")
            break


Epoch 001/100 | Train Loss: 0.6622, Acc: 0.7349 | Val Loss: 0.2009, Acc: 0.9191 | LR: 1.00e-03
Epoch 002/100 | Train Loss: 0.2081, Acc: 0.9218 | Val Loss: 0.1392, Acc: 0.9422 | LR: 1.00e-03
Epoch 003/100 | Train Loss: 0.1551, Acc: 0.9396 | Val Loss: 0.1169, Acc: 0.9585 | LR: 1.00e-03
Epoch 004/100 | Train Loss: 0.1413, Acc: 0.9424 | Val Loss: 0.1045, Acc: 0.9579 | LR: 1.00e-03
Epoch 005/100 | Train Loss: 0.1430, Acc: 0.9395 | Val Loss: 0.1074, Acc: 0.9565 | LR: 1.00e-03
Epoch 006/100 | Train Loss: 0.1291, Acc: 0.9486 | Val Loss: 0.1134, Acc: 0.9545 | LR: 1.00e-03
Epoch 007/100 | Train Loss: 0.1361, Acc: 0.9475 | Val Loss: 0.1021, Acc: 0.9592 | LR: 1.00e-03
Epoch 008/100 | Train Loss: 0.1265, Acc: 0.9475 | Val Loss: 0.0993, Acc: 0.9592 | LR: 1.00e-03
Epoch 009/100 | Train Loss: 0.1286, Acc: 0.9475 | Val Loss: 0.0935, Acc: 0.9646 | LR: 1.00e-03
Epoch 010/100 | Train Loss: 0.1218, Acc: 0.9476 | Val Loss: 0.1079, Acc: 0.9545 | LR: 1.00e-03
Epoch 011/100 | Train Loss: 0.1229, Acc: 0.9480 | 

In [6]:

# ============================================================
# 5. TEST EVALUATION
# ============================================================
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

test_loss, test_total = 0.0, 0
all_preds, all_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        test_loss += loss.item() * xb.size(0)
        test_total += xb.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_targets.extend(yb.cpu().numpy())

test_loss /= test_total
y_pred = np.array(all_preds)
y_true = np.array(all_targets)

acc = accuracy_score(y_true, y_pred)
m_prec, m_rec, m_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
_, _, w_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")

print("\n" + "=" * 70)
print("PYTORCH INCEPTION + BiLSTM TEST RESULTS")
print("=" * 70)
print(f"Test Loss       : {test_loss:.4f}")
print(f"Accuracy        : {acc:.4f}")
print(f"Macro Precision : {m_prec:.4f}")
print(f"Macro Recall    : {m_rec:.4f}")
print(f"Macro F1        : {m_f1:.4f}")
print(f"Weighted F1     : {w_f1:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=ACTIVITY_LABELS, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))



PYTORCH INCEPTION + BiLSTM TEST RESULTS
Test Loss       : 0.3167
Accuracy        : 0.9342
Macro Precision : 0.9345
Macro Recall    : 0.9357
Macro F1        : 0.9349
Weighted F1     : 0.9340

Classification Report:

                    precision    recall  f1-score   support

           WALKING     0.9980    0.9980    0.9980       496
  WALKING_UPSTAIRS     0.9758    0.9427    0.9590       471
WALKING_DOWNSTAIRS     0.9392    0.9929    0.9653       420
           SITTING     0.8378    0.8310    0.8344       491
          STANDING     0.8582    0.8534    0.8558       532
            LAYING     0.9981    0.9963    0.9972       537

          accuracy                         0.9342      2947
         macro avg     0.9345    0.9357    0.9349      2947
      weighted avg     0.9342    0.9342    0.9340      2947

Confusion Matrix:
 [[495   0   1   0   0   0]
 [  0 444  26   1   0   0]
 [  1   2 417   0   0   0]
 [  0   7   0 408  75   1]
 [  0   0   0  78 454   0]
 [  0   2   0   0   0 535]]